In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Try to load a dataset; if not available, create a dummy one.
try:
    # Replace 'your_kaggle_dataset.csv' with the actual path to your Kaggle dataset.
    # For example, if you're using the Titanic dataset, it might be 'titanic/train.csv'
    df = pd.read_csv('your_kaggle_dataset.csv')
    print("Dataset loaded successfully from 'your_kaggle_dataset.csv'.")
    # Basic check for target column (assuming 'Survived' for Titanic-like data)
    if 'Survived' in df.columns:
        target_column = 'Survived'
    elif 'target' in df.columns:
        target_column = 'target'
    else:
        print("Warning: No common target column ('Survived' or 'target') found. Please define your target column.")
        target_column = None # User needs to define this

except FileNotFoundError:
    print("File 'your_kaggle_dataset.csv' not found. Creating a dummy dataset for demonstration.")
    # Create a dummy dataset for demonstration if no file is found
    data = {
        'feature_1': np.random.rand(100),
        'feature_2': np.random.randint(0, 5, 100),
        'feature_3': np.random.randn(100),
        'categorical_feature': np.random.choice(['A', 'B', 'C'], 100),
        'target': np.random.randint(0, 2, 100) # Binary classification target
    }
    df = pd.DataFrame(data)
    target_column = 'target'

print("\nFirst 5 rows of the dataset:")
display(df.head())
print("\nDataset Info:")
df.info()

File 'your_kaggle_dataset.csv' not found. Creating a dummy dataset for demonstration.

First 5 rows of the dataset:


,feature_1,feature_2,feature_3,categorical_feature,target
0,0.527956,4,0.086469,A,1
1,0.618458,2,0.203724,C,0
2,0.460112,3,1.499845,C,0
3,0.065761,0,-0.035807,B,1
4,0.775613,1,-2.032444,A,0



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   feature_1            100 non-null    float64
 1   feature_2            100 non-null    int64  
 2   feature_3            100 non-null    float64
 3   categorical_feature  100 non-null    object 
 4   target               100 non-null    int64  
dtypes: float64(2), int64(2), object(1)
memory usage: 4.0+ KB


In [4]:
if target_column is None:
    print("Error: Target column not defined. Please manually specify it, e.g., target_column = 'your_column_name'")
else:
    # Separate features (X) and target (y)
    X = df.drop(columns=[target_column])
    y = df[target_column]

    # Identify numerical and categorical columns
    numerical_cols = X.select_dtypes(include=np.number).columns
    categorical_cols = X.select_dtypes(include='object').columns

    # Handle missing values
    for col in numerical_cols:
        if X[col].isnull().sum() > 0:
            X[col] = X[col].fillna(X[col].mean())
            print(f"Filled missing values in numerical column '{col}' with mean.")

    for col in categorical_cols:
        if X[col].isnull().sum() > 0:
            X[col] = X[col].fillna(X[col].mode()[0])
            print(f"Filled missing values in categorical column '{col}' with mode.")

    # One-hot encode categorical features
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
    print("\nFeatures after one-hot encoding:")
    display(X.head())

    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

    print(f"\nTraining set size: {len(X_train)} samples")
    print(f"Testing set size: {len(X_test)} samples")


Features after one-hot encoding:


,feature_1,feature_2,feature_3,categorical_feature_B,categorical_feature_C
0,0.527956,4,0.086469,False,False
1,0.618458,2,0.203724,False,True
2,0.460112,3,1.499845,False,True
3,0.065761,0,-0.035807,True,False
4,0.775613,1,-2.032444,False,False



Training set size: 70 samples
Testing set size: 30 samples


In [5]:
from sklearn.ensemble import RandomForestClassifier

if target_column is not None:
    # Initialize and train the Random Forest Classifier
    # n_estimators: number of trees in the forest
    # random_state: for reproducibility
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    print("\nTraining Random Forest Classifier...")
    rf_model.fit(X_train, y_train)

    # Make predictions on the test set
    y_pred_rf = rf_model.predict(X_test)

    # Evaluate the model
    print("\n--- Random Forest (Bagging) Model Evaluation ---")
    print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
    print("Classification Report:")
    print(classification_report(y_test, y_pred_rf))
else:
    print("Skipping Bagging implementation: Target column not defined.")


Training Random Forest Classifier...

--- Random Forest (Bagging) Model Evaluation ---
Accuracy: 0.6333
Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.59      0.65        17
           1       0.56      0.69      0.62        13

    accuracy                           0.63        30
   macro avg       0.64      0.64      0.63        30
weighted avg       0.65      0.63      0.63        30



In [6]:
from sklearn.ensemble import GradientBoostingClassifier

if target_column is not None:
    # Initialize and train the Gradient Boosting Classifier
    # n_estimators: number of boosting stages
    # learning_rate: shrinks the contribution of each tree
    # random_state: for reproducibility
    gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
    print("\nTraining Gradient Boosting Classifier...")
    gb_model.fit(X_train, y_train)

    # Make predictions on the test set
    y_pred_gb = gb_model.predict(X_test)

    # Evaluate the model
    print("\n--- Gradient Boosting (Boosting) Model Evaluation ---")
    print(f"Accuracy: {accuracy_score(y_test, y_pred_gb):.4f}")
    print("Classification Report:")
    print(classification_report(y_test, y_pred_gb))
else:
    print("Skipping Boosting implementation: Target column not defined.")


Training Gradient Boosting Classifier...

--- Gradient Boosting (Boosting) Model Evaluation ---
Accuracy: 0.4667
Classification Report:
              precision    recall  f1-score   support

           0       0.53      0.47      0.50        17
           1       0.40      0.46      0.43        13

    accuracy                           0.47        30
   macro avg       0.47      0.47      0.46        30
weighted avg       0.48      0.47      0.47        30

